# Google Play Store Choropleth Map Visualization (Task 2)

This notebook implements the following workflow:
1. Loads the Google Play Store dataset.
2. Cleans the installation column (removes commas and '+' signs, handles 'Free' rows).
3. Filters out categories starting with letters **'A'**, **'C'**, **'G'**, or **'S'** (case-insensitive).
4. Extracts the **top 5** app categories by total installs.
5. Maps these categories to representative countries using ISO Alpha-3 codes.
6. Renders an interactive Plotly Choropleth map where regions with installs > 1,000,000 are highlighted with custom gold borders.
7. Implements timezone-aware IST time-gating (6 PM to 8 PM) with a `TEST_MODE` bypass.
8. Exports the interactive HTML map and a static PNG map image.
9. Implements a premium Tkinter desktop dashboard class that displays the map when the time-gate is open (or in `TEST_MODE`).

In [1]:
# Configuration and Toggles
# Set TEST_MODE = True to bypass the 6 PM - 8 PM IST time restriction for verification.
# Set TEST_MODE = False to enforce strict time-gating.
TEST_MODE = True

## Step 1: Import Dependencies

In [2]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import pytz
from datetime import datetime
import tkinter as tk
from tkinter import ttk, messagebox
from PIL import Image, ImageTk

## Step 2: Load and Clean the Dataset

In [3]:
# Load the Play Store dataset
data_path = 'Dataset/Play Store Data.csv'
df = pd.read_csv(data_path)

# Clean the Installs column: remove '+' and ',', handle malformed entries
df = df[df['Installs'] != 'Free']  # Remove the header/malformed row
df['Installs_clean'] = df['Installs'].astype(str).str.replace('+', '', regex=False).str.replace(',', '', regex=False)
df['Installs_clean'] = pd.to_numeric(df['Installs_clean'], errors='coerce').fillna(0).astype(int)

# Filter out categories starting with 'A', 'C', 'G', or 'S' (case-insensitive)
# We check both uppercase and lowercase to prevent any case mismatch issues
df = df[~df['Category'].str.upper().str.startswith(('A', 'C', 'G', 'S'), na=True)]

print("Data loaded and filtered. Shape:", df.shape)
print("Remaining categories:", df['Category'].unique())

Data loaded and filtered. Shape: (8160, 14)
Remaining categories: ['BEAUTY' 'BOOKS_AND_REFERENCE' 'BUSINESS' 'DATING' 'EDUCATION'
 'ENTERTAINMENT' 'EVENTS' 'FINANCE' 'FOOD_AND_DRINK' 'HEALTH_AND_FITNESS'
 'HOUSE_AND_HOME' 'LIBRARIES_AND_DEMO' 'LIFESTYLE' 'FAMILY' 'MEDICAL'
 'PHOTOGRAPHY' 'TRAVEL_AND_LOCAL' 'TOOLS' 'PERSONALIZATION' 'PRODUCTIVITY'
 'PARENTING' 'WEATHER' 'VIDEO_PLAYERS' 'NEWS_AND_MAGAZINES'
 'MAPS_AND_NAVIGATION']


## Step 3: Get Top 5 Categories & Map to Representative Countries

In [4]:
# Aggregate installs by category
cat_installs = df.groupby('Category')['Installs_clean'].sum().reset_index()
top_5_cats = cat_installs.sort_values(by='Installs_clean', ascending=False).head(5).reset_index(drop=True)

# Define standard geographical mapping because Play Store installs are global and lacks country column
category_country_mapping = {
    'PRODUCTIVITY': {'ISO': 'USA', 'Country': 'United States'},
    'TOOLS': {'ISO': 'IND', 'Country': 'India'},
    'FAMILY': {'ISO': 'DEU', 'Country': 'Germany'},
    'PHOTOGRAPHY': {'ISO': 'BRA', 'Country': 'Brazil'},
    'NEWS_AND_MAGAZINES': {'ISO': 'AUS', 'Country': 'Australia'}
}

# Add country information to the top 5 categories dataframe
top_5_cats['Country_ISO'] = top_5_cats['Category'].map(lambda x: category_country_mapping[x]['ISO'])
top_5_cats['Country_Name'] = top_5_cats['Category'].map(lambda x: category_country_mapping[x]['Country'])

# Add validation column: highlight countries where total installations exceed 1 million (1,000,000)
top_5_cats['Exceeds_1M'] = top_5_cats['Installs_clean'] > 1000000

print("Top 5 categories prepared for Choropleth visualization:")
print(top_5_cats)

Top 5 categories prepared for Choropleth visualization:
             Category  Installs_clean Country_ISO   Country_Name  Exceeds_1M
0        PRODUCTIVITY     14176091369         USA  United States        True
1               TOOLS     11452771915         IND          India        True
2              FAMILY     10258263505         DEU        Germany        True
3         PHOTOGRAPHY     10088247655         BRA         Brazil        True
4  NEWS_AND_MAGAZINES      7496317760         AUS      Australia        True


## Step 4: Generate Interactive Plotly Choropleth Map

In [5]:
# Create the interactive Choropleth Map using Plotly Express
fig = px.choropleth(
    top_5_cats,
    locations="Country_ISO",
    color="Installs_clean",
    hover_name="Category",
    hover_data={
        "Country_Name": True,
        "Installs_clean": ":,",
        "Exceeds_1M": True,
        "Country_ISO": False
    },
    color_continuous_scale=px.colors.sequential.Sunsetdark,
    labels={"Installs_clean": "Total Installs"},
    title="Global App Installs by Category (Top 5 Filtered Categories)"
)

# Apply custom styling to highlight regions where installs exceed 1M
# All top 5 categories exceed 1 million installs, so we style them with a bold gold border line
fig.update_traces(
    marker_line_color="gold",
    marker_line_width=4,
    selector=dict(type='choropleth')
)

fig.update_layout(
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type='natural earth',
        landcolor='#2D2D2D',
        oceancolor='#1A1A1A',
        showocean=True,
        bgcolor='#1E1E1E'
    ),
    paper_bgcolor='#1E1E1E',
    font_color='#E0E0E0',
    width=950,
    height=550,
    margin=dict(l=0, r=0, t=50, b=0)
)

# Create export directories if they don't exist
os.makedirs("Screenshots", exist_ok=True)
os.makedirs("Documentation", exist_ok=True)

# Save interactive HTML format for documentation folder
fig.write_html("Documentation/choropleth_map.html")

# Export a high-resolution static PNG representation for the dashboard screenshot folder
fig.write_image("Screenshots/choropleth_map.png")

print("Interactive map saved to Documentation/choropleth_map.html")
print("Static map image saved to Screenshots/choropleth_map.png")

Interactive map saved to Documentation/choropleth_map.html
Static map image saved to Screenshots/choropleth_map.png


## Step 5: Time-Gating Display Filter (6 PM - 8 PM IST)

In [6]:
# Get timezone info for IST
ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist)
hour = current_time.hour

# Graph works only between 6 PM (18:00) and 8 PM (20:00) IST
is_active_hour = (18 <= hour < 20)

print(f"Current System Time (IST): {current_time.strftime('%Y-%m-%d %I:%M:%S %p')}")
print(f"Is Active Time Window (6 PM - 8 PM IST): {is_active_hour}")

if is_active_hour or TEST_MODE:
    if TEST_MODE and not is_active_hour:
        print("[TEST MODE] Bypassing time-gating filter logic.")
    print("Rendering interactive Plotly Choropleth Map:")
    fig.show()
else:
    print("Dashboard Hidden: The Category Installs Choropleth Map is only available between 6 PM and 8 PM IST.")

Current System Time (IST): 2026-06-20 02:21:19 PM
Is Active Time Window (6 PM - 8 PM IST): False
[TEST MODE] Bypassing time-gating filter logic.
Rendering interactive Plotly Choropleth Map:


## Step 6: Tkinter App Dashboard Integration

In [7]:
class AppDashboard(tk.Tk):
    def __init__(self, test_mode=True):
        super().__init__()
        self.title("Google Play Store Analytics Dashboard - Task 2")
        self.geometry("1150x700")
        self.configure(bg="#121212")  # Ultra dark premium color
        
        self.test_mode = test_mode
        self.check_time_and_render()

    def check_time_and_render(self):
        # Fetch timezone info for IST
        ist = pytz.timezone('Asia/Kolkata')
        now_ist = datetime.now(ist)
        hour = now_ist.hour
        is_active = (18 <= hour < 20)

        # Clean current widgets
        for widget in self.winfo_children():
            widget.destroy()

        if is_active or self.test_mode:
            self.build_active_dashboard(now_ist)
        else:
            self.build_restricted_dashboard(now_ist)

    def build_active_dashboard(self, time_now):
        # Main container with responsive padding
        main_container = tk.Frame(self, bg="#121212")
        main_container.pack(fill=tk.BOTH, expand=True, padx=20, pady=20)

        # Header section
        header = tk.Frame(main_container, bg="#1E1E1E", height=90, bd=0)
        header.pack(fill=tk.X, pady=(0, 20))
        title = tk.Label(
            header, 
            text="Play Store Installs - Geographic Category Analysis", 
            font=("Outfit", 22, "bold"), 
            fg="#00E5FF", 
            bg="#1E1E1E"
        )
        title.pack(side=tk.LEFT, padx=25, pady=20)

        time_display = time_now.strftime("%I:%M %p IST")
        info_label = tk.Label(
            header, 
            text=f"Time: {time_display} | Mode: {'TEST_BYPASS' if self.test_mode else 'LIVE'}", 
            font=("Inter", 11, "italic"), 
            fg="#A0A0A0", 
            bg="#1E1E1E"
        )
        info_label.pack(side=tk.RIGHT, padx=25, pady=25)

        # Dashboard layout
        content = tk.Frame(main_container, bg="#121212")
        content.pack(fill=tk.BOTH, expand=True)

        # Sidebar for category and details
        sidebar = tk.Frame(content, bg="#1E1E1E", width=350)
        sidebar.pack(side=tk.LEFT, fill=tk.Y, padx=(0, 20))
        sidebar.pack_propagate(False)

        sidebar_title = tk.Label(
            sidebar, 
            text="Top 5 Category Metrics", 
            font=("Outfit", 16, "bold"), 
            fg="#E0E0E0", 
            bg="#1E1E1E"
        )
        sidebar_title.pack(anchor="w", padx=20, pady=20)

        # Add list of categories and installs
        for idx, row in top_5_cats.iterrows():
            cat_frame = tk.Frame(sidebar, bg="#2D2D2D", pady=8, padx=12, bd=0)
            cat_frame.pack(fill=tk.X, padx=20, pady=6)
            
            cat_name = tk.Label(
                cat_frame, 
                text=f"{row['Category']}", 
                font=("Inter", 11, "bold"), 
                fg="#00E5FF", 
                bg="#2D2D2D"
            )
            cat_name.pack(anchor="w")
            
            cat_desc = tk.Label(
                cat_frame, 
                text=f"Installs: {row['Installs_clean']:,} | mapped to {row['Country_Name']}", 
                font=("Inter", 9), 
                fg="#B0B0B0", 
                bg="#2D2D2D"
            )
            cat_desc.pack(anchor="w", pady=(4, 0))

        # Highlight Info Alert Box
        alert_box = tk.Frame(sidebar, bg="#2A1D0E", highlightbackground="#FFB300", highlightthickness=1)
        alert_box.pack(fill=tk.X, padx=20, pady=25)
        
        alert_text = tk.Label(
            alert_box,
            text="⚠️ HIGH INSTALL HIGHLIGHT:\nCountries styled with a thick gold border have app installations exceeding 1 Million.\nAll top 5 categories meet this threshold.",
            font=("Inter", 9, "bold"),
            fg="#FFB300",
            bg="#2A1D0E",
            justify="left",
            wraplength=270,
            padx=10,
            pady=10
        )
        alert_text.pack(fill=tk.BOTH)

        # Right side panel (displays map)
        map_container = tk.Frame(content, bg="#1E1E1E")
        map_container.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True)

        map_header = tk.Label(
            map_container,
            text="Interactive Geographic Map Visualization (Plotly Export)",
            font=("Outfit", 15, "bold"),
            fg="#E0E0E0",
            bg="#1E1E1E"
        )
        map_header.pack(pady=15)

        # Render map image
        map_path = "Screenshots/choropleth_map.png"
        if os.path.exists(map_path):
            try:
                img = Image.open(map_path)
                # Fit to dimensions
                img = img.resize((710, 420), Image.Resampling.LANCZOS)
                self.map_img_tk = ImageTk.PhotoImage(img)

                img_lbl = tk.Label(map_container, image=self.map_img_tk, bg="#1E1E1E")
                img_lbl.pack(padx=20, pady=10, expand=True, fill=tk.BOTH)
            except Exception as e:
                err_lbl = tk.Label(
                    map_container,
                    text=f"Error loading image asset: {str(e)}",
                    font=("Inter", 12),
                    fg="#FF5252",
                    bg="#1E1E1E"
                )
                err_lbl.pack(expand=True)
        else:
            missing_lbl = tk.Label(
                map_container,
                text="Visualization image asset not found.\nPlease compile the Plotly cell to write the asset.",
                font=("Inter", 12),
                fg="#FFB300",
                bg="#1E1E1E"
            )
            missing_lbl.pack(expand=True)

    def build_restricted_dashboard(self, time_now):
        # Layout for unauthorized time
        frame = tk.Frame(self, bg="#121212")
        frame.pack(fill=tk.BOTH, expand=True)

        lock_lbl = tk.Label(
            frame,
            text="🔒 Dashboard Visuals Offline",
            font=("Outfit", 28, "bold"),
            fg="#FF5252",
            bg="#121212"
        )
        lock_lbl.pack(expand=True, pady=(180, 10))

        time_display = time_now.strftime("%I:%M:%S %p IST")
        desc_lbl = tk.Label(
            frame,
            text=f"The Category Installs Choropleth Map is only visible between 6:00 PM and 8:00 PM IST.\nCurrent local time: {time_display}.\n\n(To preview this dashboard now, initialize with `test_mode=True`.)",
            font=("Inter", 13),
            fg="#A0A0A0",
            bg="#121212",
            justify="center"
        )
        desc_lbl.pack(expand=True, pady=(0, 180))

# To launch the dashboard window from Jupyter, uncomment the lines below and run the cell:
# app = AppDashboard(test_mode=TEST_MODE)
# app.mainloop()